**data**

In [35]:
reviews_main = [
    "The product quality is excellent and worth the price",
    "Delivery was very late and disappointing",
    "Packaging was damaged but product works fine",
    "Affordable and useful for daily use",
    "Customer service is very slow",
    "Design is beautiful and performance is great",
    "Not worth the money",
    "Value for money product",
    "Battery life is too short",
    "Very satisfied with the purchase"
]

**Requried Libraries**

In [ ]:
!pip install groq
from groq import Groq
import json
from google.colab import userdata
import os

**Setup Envirnonment**

In [42]:
os.environ["GROQ_API_KEY"] = userdata.get('Groq_class')
client = Groq(api_key=os.environ["GROQ_API_KEY"])

**LLM 'llama' model Building**

In [88]:
def analyze_reviews(reviews):
    reviews_text = "\n".join(reviews)
    prompt = f""" You are an AI assistant that analyzes product reviews.

              Return output in STRICT JSON format:
              {{
                "summary": "Write a short, natural 1-sentence summary without phrases like 'the product received a review".
                "pros": [],
                "cons": [],
                "sentiment": "Positive/Negative/Mixed/Neutral",
                "confidence": 0.0
              }}

              Rules:
              - Output MUST be valid JSON
              - Confidence should be between 0 and 1
              - If unsure, reduce confidence
              - If reviews are unclear, return Neutral with low confidence
              - Correct minor spelling mistakes in output
              - Only use given reviews
              - Be concise

              Reviews:
              {reviews_text}
              """

    response = client.chat.completions.create(model="llama-3.1-8b-instant",
                                              messages=[{"role": "user", "content": prompt}],
                                              temperature=0.3)

    return response.choices[0].message.content

# Safe JSON parsing
def safe_parse(output):
    try:
        return json.loads(output)
    except Exception as e:
        print("JSON parsing failed:", e)
        return None

# Validation function
def validate_output(data):

    if data is None:
        print("No data to validate")
        return False

    required_keys = ["summary", "pros", "cons", "sentiment", "confidence"]

    for key in required_keys:
        if key not in data:
            print(f"{key} missing")
            return False

    if not isinstance(data["pros"], list):
        print("pros should be list")
        return False

    if not isinstance(data["cons"], list):
        print("cons should be list")
        return False

    if data["sentiment"] not in ["Positive", "Negative", "Mixed", "Neutral"]:
        print("invalid sentiment value")
        return False

    if not (0 <= data["confidence"] <= 1):
        print("confidence should be between 0 and 1")
        return False

    print("Output is correct")
    return True
output = analyze_reviews(reviews_main)
parsed_output = safe_parse(output)

if parsed_output:
    print(parsed_output)
    validate_output(parsed_output)
else:
    print("Model returned invalid JSON. Try again.")

{'summary': 'The product received mixed reviews, with some praising its quality and performance, while others were disappointed with delivery and customer service.', 'pros': ['excellent product quality', 'worth the price', 'affordable', 'useful for daily use', 'beautiful design', 'great performance', 'value for money', 'satisfied with the purchase'], 'cons': ['delivery was very late', 'disappointing', 'packaging was damaged', 'slow customer service', 'not worth the money', 'battery life is too short'], 'sentiment': 'Mixed', 'confidence': 0.6}
Output is correct


**Test Cases - Evaluation**

In [82]:
tc1 = ["Excellent product", "Very happy", "Fast delivery"]   # Positive
tc2 = ["Good but delivery late", "Nice but packaging bad"]   # Mixed
tc3 = ["Very bad product", "Waste of money"]                 # Negative
tc4 = ["Average product", "Okay experience"]                 # Neutral
tc5 = ["Amazing quality", "Loved it"]                        # Positive
tc6 = ["Not good", "Battery bad"]                            # Negative
tc7 = ["Good but expensive"]                                 # Mixed
tc8 = ["I don't know about this product"]                    # Uncertain
tc9 = ["المنتج جيد جدا", "التوصيل متأخر"]                   # Arabic
tc10 = ["Value for money", "Good design", "Slow service"]    # Mixed

test_cases = [tc1, tc2, tc3, tc4, tc5, tc6, tc7, tc8, tc9, tc10]
expected = ["Positive", "Mixed", "Negative", "Neutral", "Positive", "Negative", "Mixed", "Neutral", "Mixed", "Mixed"]
correct = 0

for i, case in enumerate(test_cases):
    print(f"\nTest Case {i+1}")

    output = analyze_reviews(case)
    parsed = safe_parse(output)

    if parsed:
        is_valid = validate_output(parsed)

        predicted = parsed["sentiment"]
        confidence = parsed["confidence"]

        print("Expected:", expected[i])
        print("Predicted:", predicted)
        print("Confidence:", confidence)

        if predicted == expected[i]:
            correct += 1
    else:
        print("Parsing failed")

print("\nFinal Accuracy:", correct, "/", len(test_cases))


Test Case 1
Output is correct
Expected: Positive
Predicted: Positive
Confidence: 1.0

Test Case 2
Output is correct
Expected: Mixed
Predicted: Mixed
Confidence: 0.6

Test Case 3
Output is correct
Expected: Negative
Predicted: Negative
Confidence: 1.0

Test Case 4
Output is correct
Expected: Neutral
Predicted: Neutral
Confidence: 0.5

Test Case 5
Output is correct
Expected: Positive
Predicted: Positive
Confidence: 1.0

Test Case 6
Output is correct
Expected: Negative
Predicted: Negative
Confidence: 0.8

Test Case 7
Output is correct
Expected: Mixed
Predicted: Negative
Confidence: 0.8

Test Case 8
Output is correct
Expected: Neutral
Predicted: Neutral
Confidence: 0.2

Test Case 9
Output is correct
Expected: Mixed
Predicted: Mixed
Confidence: 0.6

Test Case 10
Output is correct
Expected: Mixed
Predicted: Negative
Confidence: 0.8

Final Accuracy: 8 / 10


**User Interaction with LLM**

In [90]:
reviews = []

n = int(input("Enter number of reviews: "))

for i in range(n):
    review = input(f"Enter review {i+1}: ")
    reviews.append(review)

output = analyze_reviews(reviews)

import json
parsed_output = json.loads(output)

print(parsed_output)
validate_output(parsed_output)

Enter number of reviews: 2
Enter review 1: product was good 
Enter review 2: product price is very high
{'summary': 'The product received mixed reviews.', 'pros': ['good'], 'cons': ['price is very high'], 'sentiment': 'Mixed', 'confidence': 0.5}
Output is correct


True

In [ ]:
tc1 = ["Excellent product", "Very happy", "Fast delivery"]
tc2 = ["Good but delivery late", "Nice but packaging bad"]
tc3 = ["Very bad product", "Waste of money"]

test_cases = [tc1, tc2, tc3]
expected = ["Positive", "Mixed", "Negative"]

for i, case in enumerate(test_cases):
    print(f"\nTest Case {i+1}")

    output = analyze_reviews(case)
    parsed_output = json.loads(output)

    predicted = parsed_output["sentiment"]

    print(f"Expected: {expected[i]}")
    print(f"Predicted: {predicted}")